In [1]:
import pandas as pd

# Load the CSV file into a DataFrame
df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')

# Display the first 5 rows of the dataset to get an overview
df.head()


,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [2]:
# Function to tokenize text into words
def tokenize(text):
    # Convert text to lowercase for uniformity
    text = text.lower()

    # Remove question marks from the text
    text = text.replace('?', '')

    # Remove apostrophes from the text
    text = text.replace("'", "")

    # Split the cleaned text into a list of words (tokens)
    return text.split()


In [3]:
# Example usage of the tokenize function
tokens = tokenize('What is the capital of France?')

# Output the resulting tokens list
print(tokens)  # Expected output: ['what', 'is', 'the', 'capital', 'of', 'france']


['what', 'is', 'the', 'capital', 'of', 'france']


In [4]:
# Initialize the vocabulary dictionary with a special token '<UNK>'
# '<UNK>' stands for 'unknown' and is used to represent any word not found in the vocabulary
vocab = {'<UNK>': 0}


In [5]:
# Function to build vocabulary from a dataframe row containing 'question' and 'answer' text
def build_vocab(row):
    # Tokenize question and answer texts
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])

    # Combine tokens from both question and answer
    merged_tokens = tokenized_question + tokenized_answer

    # Add tokens to the vocab dictionary if they are not already present
    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)  # Assign a unique index to each new token


In [6]:
# Apply the build_vocab function to each row in the dataframe
# This will populate the 'vocab' dictionary with all unique tokens from questions and answers
df.apply(build_vocab, axis=1)


,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [7]:
len(vocab)

324

In [8]:
# Function to convert a text string into a list of numerical indices based on the vocabulary
def text_to_indices(text, vocab):
    indexed_text = []

    # Tokenize the input text
    for token in tokenize(text):
        # Append the token's index if it exists in vocab, else append the index of <UNK> token
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])

    return indexed_text


In [9]:
# Example usage of text_to_indices function
text_to_indices("What is campusx", vocab)


[1, 2, 0]

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader

In [11]:
# Custom Dataset class for handling Question-Answer pairs
class QADataset(Dataset):

    def __init__(self, df, vocab):
        """
        Initializes the dataset.

        Args:
            df (DataFrame): The DataFrame containing 'question' and 'answer' columns.
            vocab (dict): A dictionary mapping tokens to indices.
        """
        self.df = df              # Save the dataframe
        self.vocab = vocab        # Save the vocabulary

    def __len__(self):
        """
        Returns the total number of samples in the dataset.
        """
        return self.df.shape[0]

    def __getitem__(self, index):
        """
        Retrieves the sample at the specified index.

        Args:
            index (int): The index of the sample to retrieve.

        Returns:
            tuple: (Tensor of token indices for question, Tensor of token indices for answer)
        """
        # Tokenize and convert question and answer to indices
        numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
        numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

        # Convert lists of indices to PyTorch tensors
        return torch.tensor(numerical_question), torch.tensor(numerical_answer)


In [12]:
# Create an instance of QADataset using the dataframe and vocabulary
dataset = QADataset(df, vocab)

# Wrap the dataset in a DataLoader for easy batch processing
# batch_size=1 means each batch will contain one question-answer pair
# shuffle=True randomizes the order of data each epoch
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# Iterate through the DataLoader
for question, answer in dataloader:
    # Print the question tensor and the first element of the answer tensor
    # `answer[0]` is used to remove the extra batch dimension since batch_size=1
    print(question, answer[0])


tensor([[42, 43, 44, 45, 46, 47, 48]]) tensor([49])
tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([259])
tensor([[ 78,  79, 261, 151,  14, 262, 153]]) tensor([36])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([9])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([215])
tensor([[ 10, 140,   3, 141, 142,  12, 143,  83,   3, 144]]) tensor([145])
tensor([[ 10,  75,   3, 296,  19, 297]]) tensor([298])
tensor([[10, 75, 76]]) tensor([77])
tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]]) tensor([155])
tensor([[  1,   2,   3,  33,  34,   5, 245]]) tensor([246])
tensor([[ 42, 125,   2,  62,  63,   3, 126, 127]]) tensor([128])
tensor([[ 42, 167,   2,   3,  17, 168, 169]]) tensor([170])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([52])
tensor([[ 1,  2,  3, 92, 93, 94]]) tensor([95])
tensor([[ 42, 137,   2,  62,  39,   3, 322, 323]]) tensor([6])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([74])
tensor([[10,  2,  3, 66,  5, 67]]) tensor([68])
tensor([[ 1,  2,  3, 24, 25,  5, 26, 19, 27]

In [13]:
import torch.nn as nn

In [14]:
class SimpleRNN(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        # Embedding layer: maps word indices to 50-dimensional vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim=50)

        # RNN layer: processes the sequence of embeddings
        # input_size=50 (same as embedding dim), hidden_size=64
        self.rnn = nn.RNN(50, 64, batch_first=True)

        # Fully connected layer: maps the final RNN output to vocab size
        # so we can predict the next word in the vocabulary
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, question):
        # Convert input question (batch of token indices) to embeddings
        embedded_question = self.embedding(question)

        # Pass embeddings through RNN
        # `hidden` contains output at each time step
        # `final` is the hidden state from the last time step
        hidden, final = self.rnn(embedded_question)

        # Squeeze to remove sequence dimension (needed when batch_first=True)
        # and pass final hidden state to the fully connected layer
        output = self.fc(final.squeeze(0))

        return output


In [15]:
x = nn.Embedding(324, embedding_dim=50)  # Embedding layer: 324 words → 50-dim vectors
y = nn.RNN(50, 64, batch_first=True)     # RNN layer: input 50-dim → hidden 64-dim
z = nn.Linear(64, 324)                   # Linear layer: output logits over 324 vocabulary items

a = dataset[0][0].reshape(1, 6)          # Take 1st question, reshape to (1, 6) => (batch_size=1, sequence_length=6)
print("shape of a:", a.shape)
# Output: shape of a: torch.Size([1, 6])

b = x(a)                                 # Embedding: input (1, 6) → output (1, 6, 50)
print("shape of b:", b.shape)
# Output: shape of b: torch.Size([1, 6, 50])

c, d = y(b)                              # RNN: input (1, 6, 50) → c: output at all timesteps, d: final hidden state
print("shape of c:", c.shape)           # shape of c: (1, 6, 64) → output for all 6 time steps
print("shape of d:", d.shape)           # shape of d: (1, 1, 64) → final hidden state for the batch

e = z(d.squeeze(0))                      # Squeeze batch dimension (1, 64) → Linear → (1, 324)
print("shape of e:", e.shape)
# Output: shape of e: torch.Size([1, 324])


shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [16]:
learning_rate = 0.001  # Learning rate for optimizer
epochs = 20            # Number of times the model will see the full dataset

# Initialize the model with the size of the vocabulary
model = SimpleRNN(len(vocab))

# Define the loss function
criterion = nn.CrossEntropyLoss()
# - This is suitable for multi-class classification tasks
# - It expects raw logits (not softmaxed) and target class indices

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
# - Adam is a popular optimizer that adjusts learning rates adaptively


In [17]:
# Training loop for SimpleRNN model
for epoch in range(epochs):

    total_loss = 0  # To accumulate total loss per epoch

    for question, answer in dataloader:

        # 1. Reset gradients from previous iteration
        optimizer.zero_grad()

        # 2. Forward pass: compute predicted outputs by passing question to the model
        output = model(question)  # Output shape: (1, vocab_size)

        # 3. Compute loss: compares predicted output vs actual answer index
        loss = criterion(output, answer[0])  # answer[0] removes batch dimension

        # 4. Backward pass: compute gradients
        loss.backward()

        # 5. Update model weights
        optimizer.step()

        # 6. Accumulate loss for reporting
        total_loss += loss.item()

    # Print average loss for the current epoch
    print(f"Epoch: {epoch+1}, Loss: {total_loss:.4f}")


Epoch: 1, Loss: 525.3505
Epoch: 2, Loss: 457.1098
Epoch: 3, Loss: 378.4520
Epoch: 4, Loss: 314.5934
Epoch: 5, Loss: 262.6190
Epoch: 6, Loss: 214.0483
Epoch: 7, Loss: 170.2012
Epoch: 8, Loss: 132.1370
Epoch: 9, Loss: 101.8948
Epoch: 10, Loss: 78.5138
Epoch: 11, Loss: 61.0699
Epoch: 12, Loss: 47.5132
Epoch: 13, Loss: 37.8624
Epoch: 14, Loss: 30.9683
Epoch: 15, Loss: 25.1680
Epoch: 16, Loss: 21.0197
Epoch: 17, Loss: 17.6617
Epoch: 18, Loss: 14.9388
Epoch: 19, Loss: 12.7702
Epoch: 20, Loss: 11.1192


In [18]:
def predict(model, question, threshold=0.5):
    """
    Predict the answer token given a question using the trained model.

    Parameters:
        model: Trained SimpleRNN model.
        question (str): Input question string.
        threshold (float): Confidence threshold for prediction.

    Returns:
        None. Prints the predicted word or "I don't know".
    """

    # 1. Convert question to list of token indices using the vocabulary
    numerical_question = text_to_indices(question, vocab)

    # 2. Convert to PyTorch tensor and add batch dimension: shape -> (1, sequence_length)
    question_tensor = torch.tensor(numerical_question).unsqueeze(0)

    # 3. Feed the input tensor to the model to get raw scores (logits)
    output = model(question_tensor)  # shape -> (1, vocab_size)

    # 4. Apply softmax to get probabilities for each word in the vocab
    probs = torch.nn.functional.softmax(output, dim=1)

    # 5. Get the index of the word with the highest probability
    value, index = torch.max(probs, dim=1)

    # 6. Check if the confidence is above threshold
    if value.item() < threshold:
        print("I don't know")
        return

    # 7. Map index to word and print the predicted word
    predicted_index = index.item()
    predicted_word = list(vocab.keys())[predicted_index]
    print(predicted_word)


In [19]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [20]:
list(vocab.keys())[7]

'paris'